# 임베딩 모델 작은 샘플 비교

이 노트북은 싼타페 PDF의 일부 내용으로 한국어 특화 모델과 다국어 모델의 검색 순위를 비교합니다.

- `jhgan/ko-sroberta-multitask-mrl`: 한국어 문장 검색에 맞춰 학습한 모델입니다. 기본 벡터는 768차원입니다.
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`: 한국어를 포함한 여러 언어를 지원하며, 384차원 벡터를 만듭니다.

이 비교는 모델 선택을 위한 첫 실험입니다. 전체 PDF를 임베딩하거나 Supabase에 저장하지 않습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- [모델 추가] jhgan/ko-sroberta-multitask-mrl을 수업에도 나온 다국어 MiniLM과 비교합니다.
- 이 모델은 확인한 수업 노트북에서 찾지 못했으며 이번 개인 실험에서 추가한 후보입니다.
- 10개 샘플·27개 질문의 결과로 우선 후보를 선택했습니다. 전체 질문에서의 우수성을 확정하는 결과는 아닙니다.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


In [2]:
# [프로젝트 추가 기록] [모델 추가] jhgan/ko-sroberta-multitask-mrl을 수업에도 나온 다국어 MiniLM과 비교합니다.
# [수업 개념] MiniLM은 수업 자료에 나온 모델이고 jhgan 모델은 개인 비교 후보입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import re
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# 노트북 폴더에서 실행해도 PDF를 찾도록 프로젝트 폴더를 계산합니다.
project_folder = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
pdf_path = project_folder / 'data' / 'santafe_hev_manual.pdf'
reader = PdfReader(str(pdf_path))

print(f'PDF 경로: {pdf_path}')
print(f'PDF 전체 페이지 수: {len(reader.pages)}')

PDF 경로: C:\mle-02-p1-team1\data\santafe_hev_manual.pdf
PDF 전체 페이지 수: 779


## 1. 매뉴얼에서 비교용 글을 준비합니다

PDF 페이지 전체를 한 덩어리로 넣으면 비교 모델마다 글이 잘리는 길이가 달라질 수 있습니다. 그래서 주제별로 짧게 나눈 임시 샘플을 사용합니다. 이 길이는 최종 청킹 기준이 아닙니다. 그림이 있는 두 항목에는 앞서 확인한 그림 설명도 검색 글에 붙입니다.

In [4]:
# [프로젝트 추가 기록] [모델 추가] jhgan/ko-sroberta-multitask-mrl을 수업에도 나온 다국어 MiniLM과 비교합니다.
# [수업 개념] MiniLM은 수업 자료에 나온 모델이고 jhgan 모델은 개인 비교 후보입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
def pdf_text(page_number):
    # 사람이 세는 PDF 페이지 번호를 파이썬 페이지 목록에 맞춰 읽습니다.
    return (reader.pages[page_number - 1].extract_text() or '').strip()

# 각 문서의 글은 실제 PDF에서 가져오고, 그림 설명은 이미지 확인 결과를 덧붙입니다.
page_44 = pdf_text(44)
page_45 = pdf_text(45).replace('2C_VINPosition', '')
page_51 = pdf_text(51)
page_52 = pdf_text(52)
page_53 = pdf_text(53)
page_72 = pdf_text(72)
page_74 = pdf_text(74).replace('2C_LegSupportCautionAdjust', '')

def between(text, start, end=None):
    # 페이지에서 지정한 제목부터 다음 제목 전까지 한 주제 글을 잘라냅니다.
    start_index = text.find(start)
    if start_index == -1:
        raise ValueError(f'PDF에서 제목을 찾지 못했습니다: {start}')
    end_index = text.find(end, start_index + len(start)) if end else -1
    return text[start_index:end_index if end_index != -1 else None].strip()

tire_title = '규격 타이어 장착 및 타이어 공기압 수시 점검'
cluster_title = '클러스터 및 페달류 점검'
posture_title = '올바른 운전 자세'
seat_title = '좌석, 스티어링 휠, 미러 조정'
driver_title = '운전석 주변 점검'
tire_start = page_52.find(tire_title)
cluster_start = page_52.find(cluster_title, tire_start)

# 이미지 설명을 검색 글과 묶어, 글과 그림 내용을 함께 찾게 합니다.
records = [
    {'id': 'p44_oil', 'page': 44, 'text': page_44 + '\n그림 설명: 온도 눈금은 -30도부터 50도까지 표시되어 있다. 1.6 T-GDi HEV 행에 0W-20이 표시되어 있다.'},
    {'id': 'p45_vin', 'page': 45, 'text': page_45 + '\n그림 설명: 화살표는 엔진과 차량 실내 사이 차체 패널의 차대번호 타각 위치를 가리킨다.'},
    {'id': 'p51_daily_check', 'page': 51, 'text': page_51},
    {'id': 'p52_coolant', 'page': 52, 'text': page_52[:tire_start].strip() if tire_start >= 0 else page_52},
    {'id': 'p52_tire', 'page': 52, 'text': page_52[tire_start:cluster_start].strip() if tire_start >= 0 and cluster_start >= 0 else page_52},
    {'id': 'p53_posture', 'page': 53, 'text': between(page_53, posture_title, seat_title)},
    {'id': 'p53_mirrors', 'page': 53, 'text': between(page_53, seat_title, driver_title)},
    {'id': 'p53_driver_area', 'page': 53, 'text': between(page_53, driver_title)},
    {'id': 'p72_lumbar_support', 'page': 72, 'text': between(page_72, '허리 지지대(럼버 서포트) 조절하기', '시트 및 안전 장치')},
    {'id': 'p74_leg_support', 'page': 74, 'text': between(page_74, '다리 받침대를 올리거나 내릴 때', '릴렉션 기능') + '\n그림 설명: 좌석의 조절부와 번호별 움직임 방향이 표시되어 있다.'},
]

print(f'비교에 사용할 주제별 글 수: {len(records)}')
for record in records:
    print(f"PDF {record['page']}쪽 | {record['id']} | {len(record['text'])}자")

비교에 사용할 주제별 글 수: 10
PDF 44쪽 | p44_oil | 144자
PDF 45쪽 | p45_vin | 232자
PDF 51쪽 | p51_daily_check | 176자
PDF 52쪽 | p52_coolant | 399자
PDF 52쪽 | p52_tire | 328자
PDF 53쪽 | p53_posture | 203자
PDF 53쪽 | p53_mirrors | 318자
PDF 53쪽 | p53_driver_area | 204자
PDF 72쪽 | p72_lumbar_support | 349자
PDF 74쪽 | p74_leg_support | 319자


## 2. 실제 질문으로 검색 결과를 비교합니다

질문 옆의 정답 페이지는 매뉴얼을 보고 미리 정한 기준입니다. 모델이 정답 페이지의 글을 가장 높은 순위로 찾는지 비교합니다. 문서가 10개뿐인 첫 실험이므로 이 결과만으로 최종 성능을 단정하지는 않습니다.

In [6]:
# [프로젝트 추가 기록] [모델 추가] jhgan/ko-sroberta-multitask-mrl을 수업에도 나온 다국어 MiniLM과 비교합니다.
# [수업 개념] MiniLM은 수업 자료에 나온 모델이고 jhgan 모델은 개인 비교 후보입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
questions = [
    # PDF 44쪽: 표 이미지까지 설명 문장으로 연결해 검색합니다.
    {'question': '1.6 T-GDi HEV에 권장되는 엔진 오일 점도는 무엇인가요?', 'expected_page': 44},
    {'question': '이 차는 어떤 SAE 엔진 오일을 사용하나요?', 'expected_page': 44},
    # PDF 45쪽: 설명 글과 그림 설명을 함께 찾습니다.
    {'question': '차대번호는 차량의 어디에 새겨져 있나요?', 'expected_page': 45},
    {'question': '차대번호는 차량 등록에 어떤 역할을 하나요?', 'expected_page': 45},
    # PDF 51쪽: 운전 전 일상 점검입니다.
    {'question': '운전하기 전에 매일 점검해야 할 항목은 무엇인가요?', 'expected_page': 51},
    {'question': '차량 하부에 액체가 새는지 언제 확인해야 하나요?', 'expected_page': 51},
    {'question': '엔진룸을 열어서 확인해야 하는 것들은 무엇인가요?', 'expected_page': 51},
    # PDF 52쪽: 엔진룸 점검과 타이어 점검을 각각 묶었습니다.
    {'question': '브레이크액이 부족하면 어떻게 해야 하나요?', 'expected_page': 52},
    {'question': '냉각수 누수는 어느 부분에서 확인하나요?', 'expected_page': 52},
    {'question': '냉각수 보조 탱크의 적정 냉각수량은 어느 범위인가요?', 'expected_page': 52},
    {'question': '냉각수가 뜨거울 때 보조 탱크 뚜껑을 열어도 되나요?', 'expected_page': 52},
    {'question': '고속 주행 전에 타이어에서 무엇을 점검해야 하나요?', 'expected_page': 52},
    {'question': '타이어 공기압이 부족한 상태로 빠르게 달리면 어떤 현상이 생기나요?', 'expected_page': 52},
    {'question': '스탠딩 웨이브 현상은 무엇을 뜻하나요?', 'expected_page': 52},
    # PDF 53쪽: 바른 자세와 출발 전 운전석 점검입니다.
    {'question': '브레이크를 끝까지 밟았을 때 무릎은 어떤 자세여야 하나요?', 'expected_page': 53},
    {'question': '헤드레스트 높이는 운전자 신체의 어디에 맞추나요?', 'expected_page': 53},
    {'question': '주행 중에 좌석이나 미러를 조절해도 되나요?', 'expected_page': 53},
    {'question': '출발하기 전에 미러를 어떻게 조정해야 하나요?', 'expected_page': 53},
    {'question': '빈 깡통이나 물건을 페달 근처에 두면 왜 위험한가요?', 'expected_page': 53},
    {'question': '바닥 매트는 어떤 조건을 갖춰야 하나요?', 'expected_page': 53},
    {'question': '차 안에 화물을 어느 높이까지 실을 수 있나요?', 'expected_page': 53},
    # PDF 72쪽: 운전석 허리 지지대 조작입니다.
    {'question': '운전석 허리 지지대를 위아래로 움직이려면 어떻게 하나요?', 'expected_page': 72},
    {'question': '허리 받침 부분을 앞으로 나오게 하려면 어느 스위치를 누르나요?', 'expected_page': 72},
    # PDF 74쪽: 다리 받침대 사용 시 주의사항입니다.
    {'question': '다리 받침대를 움직일 때 손이나 발이 끼지 않게 하려면 무엇을 주의해야 하나요?', 'expected_page': 74},
    {'question': '아이가 다리 받침대 조절 스위치를 눌러도 되나요?', 'expected_page': 74},
    {'question': '다리 받침대 위에 앉거나 무거운 짐을 올려도 되나요?', 'expected_page': 74},
    {'question': '다리 받침대를 쓰기 전에 좌석을 어느 방향으로 옮기나요?', 'expected_page': 74},
]

model_names = {
    '한국어 특화': 'jhgan/ko-sroberta-multitask-mrl',
    '다국어 MiniLM': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
}

def normalized(vectors):
    # 코사인 유사도를 비교할 수 있도록 벡터 길이를 1로 맞춥니다.
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(lengths, 1e-12)

comparison = {}
for label, model_name in model_names.items():
    print(f'\n[{label}] 모델을 불러옵니다: {model_name}')
    model = SentenceTransformer(model_name)
    document_vectors = normalized(model.encode([item['text'] for item in records], convert_to_numpy=True, show_progress_bar=False))
    question_vectors = normalized(model.encode([item['question'] for item in questions], convert_to_numpy=True, show_progress_bar=False))
    similarities = question_vectors @ document_vectors.T
    results = []
    for question, scores in zip(questions, similarities):
        ranked = np.argsort(scores)[::-1]
        top_item = records[int(ranked[0])]
        target_ranks = [rank + 1 for rank, index in enumerate(ranked) if records[int(index)]['page'] == question['expected_page']]
        results.append({'question': question['question'], 'expected_page': question['expected_page'], 'top_page': top_item['page'], 'top_id': top_item['id'], 'target_rank': target_ranks[0] if target_ranks else None, 'top3_hit': any(records[int(index)]['page'] == question['expected_page'] for index in ranked[:3])})
    comparison[label] = {'dimension': model.get_embedding_dimension(), 'results': results}
    top1 = sum(result['top_page'] == result['expected_page'] for result in results)
    top3 = sum(result['top3_hit'] for result in results)
    print(f"벡터 차원: {model.get_embedding_dimension()} | 정답 페이지 1위: {top1}/{len(results)} | 정답 페이지 상위 3개 포함: {top3}/{len(results)}")
    for result in results:
        mark = '정답' if result['top_page'] == result['expected_page'] else '다른 페이지'
        print(f"- {result['question']}\n  정답 PDF {result['expected_page']}쪽 | 1위 PDF {result['top_page']}쪽 ({mark}) | 정답 순위: {result['target_rank']}위")

del model

print('\n이 결과는 샘플 비교용이며, 아직 Supabase에는 아무것도 저장하지 않았습니다.')


[한국어 특화] 모델을 불러옵니다: jhgan/ko-sroberta-multitask-mrl
벡터 차원: 768 | 정답 페이지 1위: 24/27 | 정답 페이지 상위 3개 포함: 27/27
- 1.6 T-GDi HEV에 권장되는 엔진 오일 점도는 무엇인가요?
  정답 PDF 44쪽 | 1위 PDF 44쪽 (정답) | 정답 순위: 1위
- 이 차는 어떤 SAE 엔진 오일을 사용하나요?
  정답 PDF 44쪽 | 1위 PDF 44쪽 (정답) | 정답 순위: 1위
- 차대번호는 차량의 어디에 새겨져 있나요?
  정답 PDF 45쪽 | 1위 PDF 45쪽 (정답) | 정답 순위: 1위
- 차대번호는 차량 등록에 어떤 역할을 하나요?
  정답 PDF 45쪽 | 1위 PDF 45쪽 (정답) | 정답 순위: 1위
- 운전하기 전에 매일 점검해야 할 항목은 무엇인가요?
  정답 PDF 51쪽 | 1위 PDF 51쪽 (정답) | 정답 순위: 1위
- 차량 하부에 액체가 새는지 언제 확인해야 하나요?
  정답 PDF 51쪽 | 1위 PDF 51쪽 (정답) | 정답 순위: 1위
- 엔진룸을 열어서 확인해야 하는 것들은 무엇인가요?
  정답 PDF 51쪽 | 1위 PDF 51쪽 (정답) | 정답 순위: 1위
- 브레이크액이 부족하면 어떻게 해야 하나요?
  정답 PDF 52쪽 | 1위 PDF 52쪽 (정답) | 정답 순위: 1위
- 냉각수 누수는 어느 부분에서 확인하나요?
  정답 PDF 52쪽 | 1위 PDF 52쪽 (정답) | 정답 순위: 1위
- 냉각수 보조 탱크의 적정 냉각수량은 어느 범위인가요?
  정답 PDF 52쪽 | 1위 PDF 52쪽 (정답) | 정답 순위: 1위
- 냉각수가 뜨거울 때 보조 탱크 뚜껑을 열어도 되나요?
  정답 PDF 52쪽 | 1위 PDF 52쪽 (정답) | 정답 순위: 1위
- 고속 주행 전에 타이어에서 무엇을 점검해야 하나요?
  정답 PDF 52쪽 | 1위 PDF 52쪽 (정답) | 정답 순위: 1위
- 타이어 공기압이 부족한 상태로 빠

## 다음 단계: 한국어 모델로 샘플 임베딩 만들기

비교 결과를 바탕으로 한국어 특화 모델을 우선 후보로 사용합니다. 이 셀은 샘플 10개를 숫자 벡터로 바꾸고 메모리에만 보관합니다. PDF 파일이나 Supabase에는 저장하지 않습니다. 벡터의 숫자 한 개가 뜻을 직접 설명하는 것은 아니며, 질문과 매뉴얼 글의 의미가 비슷한지 계산할 때 사용합니다.

In [8]:
# [프로젝트 추가 기록] [모델 추가] jhgan/ko-sroberta-multitask-mrl을 수업에도 나온 다국어 MiniLM과 비교합니다.
# [수업 개념] MiniLM은 수업 자료에 나온 모델이고 jhgan 모델은 개인 비교 후보입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
# 한국어 샘플 비교에서 우선 후보로 정한 모델을 불러옵니다.
selected_model_name = model_names['한국어 특화']
selected_model = SentenceTransformer(selected_model_name)

# 각 글을 같은 차원의 숫자 벡터로 바꿉니다.
# 벡터 길이를 1로 맞추면 나중에 코사인 유사도로 비교하기 편합니다.
sample_vectors = selected_model.encode(
    [record['text'] for record in records],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

# 출처 페이지와 검색 글, 임베딩 벡터를 메모리 안에서 한 묶음으로 관리합니다.
manual_page_numbers = {44: 39, 45: 40, 51: 46, 52: 47, 53: 48, 72: 67, 74: 69}
sample_embedding_records = []
for record, vector in zip(records, sample_vectors):
    sample_embedding_records.append({
        'content': record['text'],
        'metadata': {
            'pdf_page_number': record['page'],
            'manual_page_number': manual_page_numbers[record['page']],
            'content_type': 'sample_text_with_image_description' if record['id'] in {'p44_oil', 'p45_vin', 'p74_leg_support'} else 'sample_instruction_text',
        },
        'embedding': vector,
    })

# 벡터 개수와 크기, 첫 벡터의 일부 숫자만 확인합니다.
first_vector = sample_embedding_records[0]['embedding']
print(f'임베딩한 샘플 수: {len(sample_embedding_records)}')
print(f'모델 이름: {selected_model_name}')
print(f'벡터 차원: {len(first_vector)}')
print(f'첫 벡터의 길이: {np.linalg.norm(first_vector):.3f}')
print(f'첫 벡터 숫자 5개: {np.round(first_vector[:5], 4).tolist()}')
print(f'첫 기록의 출처: {sample_embedding_records[0]["metadata"]}')
print('저장 위치: 메모리만 사용 (파일/Supabase 저장 안 함)')

del selected_model

임베딩한 샘플 수: 10
모델 이름: jhgan/ko-sroberta-multitask-mrl
벡터 차원: 768
첫 벡터의 길이: 1.000
첫 벡터 숫자 5개: [0.0005000000237487257, -0.05009999871253967, -0.010599999688565731, 0.08460000157356262, 0.06639999896287918]
첫 기록의 출처: {'pdf_page_number': 44, 'manual_page_number': 39, 'content_type': 'sample_text_with_image_description'}
저장 위치: 메모리만 사용 (파일/Supabase 저장 안 함)


## 첫 비교 결과와 해석

PDF에서 주제별로 나눈 글 10개와 질문 27개를 사용했습니다. 한국어 특화 모델은 정답 페이지를 24/27개에서 1위로, 27/27개에서 상위 3개 안으로 찾았습니다. 다국어 MiniLM은 각각 23/27개와 25/27개였습니다.

두 모델의 차이는 크지 않지만, 이번 샘플에서는 한국어 특화 모델이 조금 더 높은 점수를 보였습니다. 그래서 우선 후보로 정합니다. 이 모델의 기본 벡터 크기는 768차원이고 MiniLM은 384차원입니다. 전체 매뉴얼에 적용하기 전에 질문을 더 확인하고, 개인 Supabase 테이블의 벡터 크기를 768로 맞출 예정입니다.

PDF 72쪽 전체를 한 글로 넣었을 때 허리 지지대 질문이 잘못 검색되어, 관련 부분만 주제별로 나누자 한국어 특화 모델의 결과가 개선됐습니다. 최종 청킹에서도 한 페이지를 무조건 한 덩어리로 만들지 말고, 제목과 주제가 바뀌는 지점을 살펴야 합니다.